# Take-Home Bonus — Hyperparameter Tuning & Scaling
**Type:** [Take-Home Extension]  
**No runtime ceiling.** Run this offline, not during the workshop session.

This notebook covers three topics that were deliberately out of scope during the live session:

1. **HPO for LightGBM** — Optuna-based search over the lag feature pipeline
2. **HPO for NHITS** — Neural architecture and training search
3. **Scaling beyond 1,000 series** — What changes when you go to 100,000 SKUs

---

**Prerequisites:**
- Complete the workshop session through Module 8
- All artifacts in `artifacts/` must exist (run `from src.checkpointing import list_checkpoints; list_checkpoints()` to verify)
- Install Optuna: `pip install optuna`
- Set `TUNE_MODELS = True` in `config.py` or override it in the cell below

**Expected runtime:** 30–90 minutes depending on `N_TRIALS` and hardware.

---
## B.1 — Setup
**[Take-Home Extension]**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (12, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print(f"Optuna {optuna.__version__} available.")
except ImportError:
    raise ImportError(
        "Optuna not found. Install with: pip install optuna\n"
        "Optuna is not in requirements.txt — it is a take-home only dependency."
    )

import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS

from config import (
    ARTIFACT_DIR,
    PARAMS_DIR,
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts, pooled_wmape
from src.schemas import validate_forecast

# Override config toggle — set True to actually run search
# Set False to read the structure without triggering long runs
TUNE_MODELS = True   # Change to False if you just want to read the code
N_TRIALS    = 30     # Number of Optuna trials per model — increase for better results

print(f"TUNE_MODELS = {TUNE_MODELS}")
print(f"N_TRIALS    = {N_TRIALS}")
print(f"RANDOM_SEED = {RANDOM_SEED}")

---
## B.2 — Load Data
**[Take-Home Extension]**

In [ ]:
panel = load_checkpoint("03_validated_panel")

# Use micro subset for HPO — tuning on 50 series is fast and representative
top_series = (
    panel.groupby("unique_id")["y"]
    .sum()
    .sort_values(ascending=False)
    .head(MICRO_SUBSET_N)
    .index
)
micro = panel[panel["unique_id"].isin(top_series)].copy()

print(f"HPO panel: {micro['unique_id'].nunique()} series, {len(micro):,} rows")
print(f"Tuning on micro subset keeps each trial under ~30 seconds.")

---
## B.3 — HPO for LightGBM via Optuna
**[Take-Home Extension]**

We search over the parameters that have the most impact on LightGBM accuracy and generalization:
- `num_leaves` — model complexity
- `learning_rate` — step size
- `n_estimators` — number of boosting rounds
- `min_child_samples` — regularization via minimum leaf size
- `subsample` and `colsample_bytree` — stochastic regularization

Each trial runs a full cross-validation on the micro subset. The objective is pooled wMAPE.

In [ ]:
def lgb_objective(trial: optuna.Trial) -> float:
    """
    Optuna objective for LightGBM. Returns pooled wMAPE on micro subset CV.
    Lower is better.
    """
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 1.0, log=True),
        "random_state":      RANDOM_SEED,
        "n_jobs":            -1,
        "verbosity":         -1,
    }

    lags = trial.suggest_categorical("lags", ["7_14_28", "7_14_21_28", "7_28"])
    lag_list = [int(x) for x in lags.split("_")]

    mlf = MLForecast(
        models=[lgb.LGBMRegressor(**params)],
        freq="D",
        lags=lag_list,
        lag_transforms={lag_list[0]: [RollingMean(window_size=28)]},
        date_features=["dayofweek", "month"],
    )

    try:
        cv = mlf.cross_validation(
            df=micro,
            h=HORIZON,
            n_windows=N_WINDOWS,
            step_size=STEP_SIZE,
            refit=REFIT,
        )
        cv = cv.reset_index()
        score = pooled_wmape(
            cv["y"].values,
            cv["LGBMRegressor"].clip(lower=0).values
        )
        return float(score) if not np.isnan(score) else 1.0
    except Exception:
        return 1.0  # Return worst-case on failure


if TUNE_MODELS:
    print(f"Running LightGBM HPO: {N_TRIALS} trials...")
    lgb_study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

    print(f"\nBest LightGBM trial:")
    print(f"  wMAPE  : {lgb_study.best_value:.4f}")
    print(f"  Params : {lgb_study.best_params}")
else:
    print("TUNE_MODELS = False — skipping LightGBM HPO search.")
    print("Set TUNE_MODELS = True in B.1 to run the search.")

---
## B.4 — Compare Tuned vs Default LightGBM
**[Take-Home Extension]**

After the search, compare the tuned parameters against both the workshop defaults and the pre-tuned params from `params/mlforecast_lgb_tuned.json`.

In [ ]:
if TUNE_MODELS:
    # Save tuned params
    tuned_params = dict(lgb_study.best_params)
    # Reconstruct lag list from categorical
    lag_str = tuned_params.pop("lags", "7_14_28")
    tuned_lag_list = [int(x) for x in lag_str.split("_")]
    tuned_params.update({"random_state": RANDOM_SEED, "n_jobs": -1, "verbosity": -1})

    tuned_path = PARAMS_DIR / "mlforecast_lgb_optuna.json"
    with open(tuned_path, "w") as f:
        json.dump(tuned_params, f, indent=2)
    print(f"Saved Optuna-tuned params to: {tuned_path.name}")

    # Score tuned model
    mlf_tuned = MLForecast(
        models=[lgb.LGBMRegressor(**tuned_params)],
        freq="D",
        lags=tuned_lag_list,
        lag_transforms={tuned_lag_list[0]: [RollingMean(window_size=28)]},
        date_features=["dayofweek", "month"],
    )
    cv_tuned = mlf_tuned.cross_validation(
        df=micro, h=HORIZON, n_windows=N_WINDOWS,
        step_size=STEP_SIZE, refit=REFIT,
    ).reset_index()
    wmape_tuned = pooled_wmape(cv_tuned["y"].values, cv_tuned["LGBMRegressor"].clip(lower=0).values)

    # Score default model for comparison
    default_path = PARAMS_DIR / "mlforecast_lgb_default.json"
    with open(default_path) as f:
        default_params = json.load(f)
    mlf_default = MLForecast(
        models=[lgb.LGBMRegressor(**default_params)],
        freq="D",
        lags=[7, 14, 28],
        lag_transforms={7: [RollingMean(window_size=28)]},
        date_features=["dayofweek", "month"],
    )
    cv_default = mlf_default.cross_validation(
        df=micro, h=HORIZON, n_windows=N_WINDOWS,
        step_size=STEP_SIZE, refit=REFIT,
    ).reset_index()
    wmape_default = pooled_wmape(cv_default["y"].values, cv_default["LGBMRegressor"].clip(lower=0).values)

    print(f"\nLightGBM HPO Results (micro subset, {MICRO_SUBSET_N} series):")
    print(f"  Default params wMAPE : {wmape_default:.4f}")
    print(f"  Optuna tuned wMAPE   : {wmape_tuned:.4f}")
    improvement = (wmape_default - wmape_tuned) / wmape_default * 100
    print(f"  Improvement          : {improvement:+.1f}%")
    print()
    print("Note: micro-subset improvement does not guarantee full-subset improvement.")
    print("Always validate tuned params on the full workshop subset before committing.")
else:
    print("Skipped — set TUNE_MODELS = True to run comparison.")

---
## B.5 — Optuna Trial History Plot
**[Take-Home Extension]**

In [ ]:
if TUNE_MODELS:
    trials_df = lgb_study.trials_dataframe()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Trial history
    axes[0].plot(trials_df["number"], trials_df["value"],
                 color="#90CAF9", linewidth=0.8, alpha=0.7)
    best_so_far = trials_df["value"].cummin()
    axes[0].plot(trials_df["number"], best_so_far,
                 color="#7B1FA2", linewidth=1.5, label="Best so far")
    axes[0].axhline(wmape_default, color="#E53935", linestyle="--",
                    linewidth=1, label=f"Default ({wmape_default:.4f})")
    axes[0].set_xlabel("Trial")
    axes[0].set_ylabel("wMAPE")
    axes[0].set_title("LightGBM HPO — Trial History")
    axes[0].legend(fontsize=9)

    # Parameter importance
    try:
        importances = optuna.importance.get_param_importances(lgb_study)
        imp_names  = list(importances.keys())[:8]
        imp_values = [importances[k] for k in imp_names]
        axes[1].barh(imp_names, imp_values, color="#7B1FA2", alpha=0.8)
        axes[1].set_title("Parameter Importance (Fanova)")
        axes[1].set_xlabel("Importance")
        axes[1].invert_yaxis()
    except Exception:
        axes[1].text(0.5, 0.5, "Importance requires >= 4 complete trials",
                     ha="center", va="center", transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()
else:
    print("Skipped — set TUNE_MODELS = True to see trial history.")

---
## B.6 — HPO for NHITS via Optuna
**[Take-Home Extension]**

NHITS is more sensitive to hyperparameters than LightGBM. The three most impactful levers are:
- `max_steps` — training budget (primary accuracy lever)
- `input_size` — context window relative to horizon
- `learning_rate` — gradient step size

Each NHITS trial is slow relative to LightGBM. Keep `N_TRIALS` at 15–20 for a first pass, and run on GPU if available.

In [ ]:
def nhits_objective(trial: optuna.Trial) -> float:
    """
    Optuna objective for NHITS. Returns pooled wMAPE on micro subset CV.
    Each trial can take 30–90 seconds on CPU — keep N_TRIALS low.
    """
    params = {
        "h":                          HORIZON,
        "input_size":                 trial.suggest_categorical("input_size", [28, 56, 84]),
        "max_steps":                  trial.suggest_int("max_steps", 100, 500, step=100),
        "learning_rate":              trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "batch_size":                 trial.suggest_categorical("batch_size", [16, 32, 64]),
        "n_freq_downsample":          trial.suggest_categorical(
                                          "n_freq_downsample",
                                          ["2_1_1", "4_2_1", "1_1_1"]
                                      ),
        "val_check_steps":            50,
        "early_stop_patience_steps":  5,
        "random_seed":                RANDOM_SEED,
        "enable_progress_bar":        False,
        "enable_model_summary":       False,
    }

    # Convert categorical to list
    nfd_str = params.pop("n_freq_downsample")
    params["n_freq_downsample"] = [int(x) for x in nfd_str.split("_")]

    try:
        nf = NeuralForecast(
            models=[NHITS(**params)],
            freq="D",
        )
        cv = nf.cross_validation(
            df=micro,
            n_windows=N_WINDOWS,
            step_size=STEP_SIZE,
            refit=REFIT,
        ).reset_index()
        score = pooled_wmape(
            cv["y"].values,
            cv["NHITS"].clip(lower=0).values
        )
        return float(score) if not np.isnan(score) else 1.0
    except Exception:
        return 1.0


if TUNE_MODELS:
    nhits_trials = max(10, N_TRIALS // 3)  # Fewer trials — each is slow
    print(f"Running NHITS HPO: {nhits_trials} trials (reduced from {N_TRIALS} — each trial is ~30-90s)...")
    nhits_study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    nhits_study.optimize(nhits_objective, n_trials=nhits_trials, show_progress_bar=True)

    best = nhits_study.best_params
    print(f"\nBest NHITS trial:")
    print(f"  wMAPE  : {nhits_study.best_value:.4f}")
    print(f"  Params : {best}")

    # Save tuned NHITS params
    nhits_tuned_output = {
        "_comment": "Optuna-tuned NHITS params. Validated on micro subset.",
        "h":                    HORIZON,
        "input_size":           best.get("input_size", 56),
        "max_steps":            best.get("max_steps", 300),
        "learning_rate":        best.get("learning_rate", 0.001),
        "batch_size":           best.get("batch_size", 32),
        "n_freq_downsample":    [int(x) for x in best.get("n_freq_downsample", "2_1_1").split("_")],
        "val_check_steps":      50,
        "early_stop_patience_steps": 5,
        "random_seed":          RANDOM_SEED,
        "enable_progress_bar":  False,
        "enable_model_summary": False,
    }
    nhits_output_path = PARAMS_DIR / "nhits_optuna.json"
    with open(nhits_output_path, "w") as f:
        json.dump(nhits_tuned_output, f, indent=2)
    print(f"Saved Optuna-tuned NHITS params to: {nhits_output_path.name}")
else:
    print("Skipped — set TUNE_MODELS = True to run NHITS HPO.")

---
## B.7 — HPO Takeaways: What Actually Matters
**[Take-Home Extension]**

After running HPO on both models, the pattern is almost always the same:

**For LightGBM:**  
The parameter importance plot usually shows `learning_rate` and `num_leaves` as the dominant factors. `subsample` and `colsample_bytree` matter when the dataset is large and prone to overfitting. On 50–1,000 dense retail series, the default params are often within 1–2 wMAPE points of the best tuned config. **The lag structure (which lags to include) frequently matters more than any individual LightGBM hyperparameter.**

**For NHITS:**  
`max_steps` and `input_size` dominate. `learning_rate` matters on longer runs. The `n_freq_downsample` configuration affects what temporal scales the model attends to — `[4, 2, 1]` is better for data with both weekly and monthly patterns.

**The honest HPO cost calculation:**  
30 Optuna trials for LightGBM on 50 series takes roughly 15 minutes. On 1,000 series it takes hours. On 100,000 series it is not practical without distributed compute. At that scale, tuning is done on a representative sample and validated on a stratified holdout — never on the full portfolio.

---
## B.8 — Scaling Beyond the Workshop: What Changes at 100,000 SKUs
**[Take-Home Extension]**

This section is conceptual — no code to run. It maps the workshop pipeline to the production challenges that appear when you scale from 1,000 to 100,000+ series.

### B.8.1 — Data Layer

**Workshop approach:** Load a 25MB parquet from disk. Fits in RAM on any laptop.

**At 100,000 SKUs × 3 years daily:** ~75M rows, ~3GB parquet. Does not fit in a Colab notebook. Requires partitioned storage (Parquet on S3, Delta Lake, or Hive) and lazy loading.

**What to build:**
- Partitioned parquet by `store_id` or `category` — models train on one partition at a time
- Schema validation at ingestion, not just at training time
- Incremental append pattern — new data is appended to the partition, not reloaded from scratch

---

### B.8.2 — Feature Pipeline

**Workshop approach:** MLForecast computes lags on the fly during cross-validation.

**At scale:** On-the-fly lag computation is too slow and not auditable. You need:
- A feature store (Feast, Tecton, or a custom solution) that precomputes lags and rolling statistics once per day
- Versioned feature snapshots so you can reproduce any historical training run
- A backfill job that recomputes features when upstream data is corrected

**The most common failure mode:** Late data invalidates precomputed lags silently. Your model trains on lag_7 values computed from data that arrives by 6am. The 3% of SKUs with data arriving at 10am get stale lags. You will not see this in your wMAPE — you will see it in stockout rates on those SKUs.

---

### B.8.3 — Training

**Workshop approach:** Single LightGBM global model trains on all series in one call.

**At scale:** A single global model works surprisingly well up to a few hundred thousand series. Beyond that:
- Hierarchical modeling by category — one LightGBM per product category, shared feature schema
- Distributed training via Spark or Dask for embarrassingly parallel models (AutoETS per series)
- Neural models (NHITS) trained on GPU clusters with data parallelism

**NHITS-specific:** At 100,000 series × 3 years × daily, a single NeuralForecast training run takes 4–8 hours on a single GPU. Practical options: train on a stratified 10,000-series sample and validate transfer, or use spot GPU instances on a weekly retraining schedule.

---

### B.8.4 — Evaluation

**Workshop approach:** Score all models in a single `score_forecasts()` call. Results fit in a 6-row leaderboard.

**At scale:** You need segment-level evaluation:
- By product category
- By sales velocity tier (A/B/C classification)
- By store region
- By seasonal pattern type

A global wMAPE of 0.22 can hide a 0.45 wMAPE on your most volatile SKUs — exactly the ones that drive your safety stock costs. Pooled metrics are the right top-line measure, but segment breakdowns are what drive model improvement decisions.

---

### B.8.5 — Serving

**Workshop approach:** Run the notebook. Forecasts saved to a local CSV.

**At scale:** A forecast serving system has three components:
1. **Batch inference job** — runs on a schedule, writes forecasts to a database or object store
2. **Freshness monitor** — alerts if forecasts are not updated by the SLA deadline
3. **Fallback policy** — serves the last valid forecast if the current batch fails, logs the fallback event

The fallback policy is the most important and least-built component. Define it before you deploy.

---

### B.8.6 — Monitoring

**Workshop approach:** None — the workshop is a single evaluation run.

**At scale:** You need at least three monitoring layers:
1. **Input monitoring** — flag series where recent actuals have shifted dramatically from historical patterns (demand shock, assortment change, data quality issue)
2. **Prediction monitoring** — flag forecasts that are implausibly large or zero for non-zero SKUs
3. **Accuracy monitoring** — rolling wMAPE tracked week-over-week, with automated retraining trigger when it degrades beyond a threshold

The hardest monitoring problem in forecasting is distinguishing between model degradation (retrain) and demand structure change (retrain with new features). Both look identical in the accuracy metric.

---
## B.9 — Recommended Next Steps
**[Take-Home Extension]**

If you want to go deeper on any topic covered in this workshop:

**On evaluation rigor:**
- Hyndman & Koehler (2006) — "Another look at measures of forecast accuracy" — the academic foundation for why MAE/MAPE/wMAPE behave the way they do
- Gneiting & Raftery (2007) — "Strictly Proper Scoring Rules" — the theoretical basis for Interval Score

**On the Nixtla stack:**
- StatsForecast, MLForecast, NeuralForecast documentation at `nixtlaverse.nixtla.io`
- The M5 competition paper (Makridakis et al., 2022) — context for why this dataset was chosen

**On Chronos:**
- Ansari et al. (2024) — "Chronos: Learning the Language of Time Series" — the original paper
- The AutoGluon-TimeSeries documentation covers the full Chronos family including larger variants

**On production systems:**
- *Designing Machine Learning Systems* (Huyen, 2022) — Chapters 7–9 cover feature stores, model deployment, and monitoring patterns that apply directly to forecasting
- Feast documentation (`feast.dev`) — open source feature store used in production at scale

**On the book:**
- *Modern Time Series Forecasting with Python* by Jeff Tackes and Manu Joseph (Packt Publishing) covers all six models from today in full detail, including the production patterns this workshop only outlined.

---

*Good luck with the build.*  
*— Jeff & Manu*